In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42
random.seed(SEED) # Python
np.random.seed(SEED) # NumPy
torch.manual_seed(SEED) # PyTorch
torch.cuda.manual_seed(SEED) # PyTorch
torch.cuda.manual_seed_all(SEED) # PyTorch
set_seed(SEED)

# deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


Load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/sara_sms112/SequenceFeaturesData_bact.csv') #bacterial data with seqeunce features only
#data = pd.read_csv('/content/drive/MyDrive/sara_sms112/NetworkFeaturesData_bact.csv') # bacterial data with network + seqeunce features

#data =  pd.read_csv('/content/drive/MyDrive/sara_sms112/FinalDataEukXcodonw.csv') # eukaryotic data with seqeunce features only
#data = pd.read_csv('/content/drive/MyDrive/sara_sms112/FinalDataEukXcodonwXstring_buckets.csv') #eukaryotic data with codonw + string features

Balance data

In [ ]:
from sklearn.utils import resample

# Balance the data (Random undersampling of the majority class)

df_majority = data[data['Essential'] == 0]
df_minority = data[data['Essential'] == 1]

df_majority_downsampled = resample(df_majority,
                                   replace=False,    # sample without replacement
                                   n_samples=len(df_minority),    # match minority class size
                                   random_state=SEED)

# combine minority class with downsampled majority class
data_balanced = pd.concat([df_majority_downsampled, df_minority])

# shuffle the balanced dataset
data_balanced = data_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True) #try random state 45 instead of 40

Serialize data

In [16]:
def serialize_for_roberta(data):

    serialized_texts = []
    labels = []

    for _, row in data.iterrows():
        text = (

            # # Gene identity & basic info for bacterial:
            # f"Gene {row['patric_id']} from organism {row['Organism_Name']} "
            # f"is on the {'positive' if row['strand']=='+' else 'negative'} strand. "
            # f"It has a protein length of {row['length_AA']}. "
            # f"This gene is annotated as: {row['Gene_Description']}. " # comment out this row for ablation studies

            # # Gene identity & basic info for eukaryotic:
            f"Gene {row['ID']} from organism {row['Organism']} "
            f"is on the {'positive' if row['Orientation']=='plus' else 'negative'} strand. "
            f"It has a protein length of {row['length_AA']}. "
            # f"This gene is annotated as: {row['Protein_description']}. " # comment out this row for ablation studies

            # Nucleotide composition
            f"GC content is {row['GC']*100:.1f} with GC3s at {row['GC3s']*100:.1f}. "
            f"T3s at {row['T3s']*100:.1f}, C3s {row['C3s']*100:.1f}, "
            f"A3s {row['A3s']*100:.1f}, G3s {row['G3s']*100:.1f}. "

            # Protein-level properties
            f"Molecular weight is {row['MolecularWeight']:.2f} and isoelectric point is {row['IsoelectricPoint']:.2f}. "
            f"Hydropathicity (Gravy) score is {row['Gravy']:.4f}, low-complexity symmetry score {row['L_sym']}. "

            # Codon usage
            f"Codon usage metrics: Nc {row['Nc']}, CAI {row['CAI']}, CBI {row['CBI']}, Fop {row['Fop']}. "

            # Full nucleotide percentages
            f"Nucleic acid composition: Adenine {row['Adenine']*100:.1f}, Cytosine {row['Cytosine']*100:.1f}, "
            f"Guanine {row['Guanine']*100:.1f}, Thymine {row['Thymine']*100:.1f}. "

            # Amino acid composition
            f"Amino acid composition: Ala {row['A']*100:.1f}, Cys {row['C']*100:.1f}, Asp {row['D']*100:.1f}, "
            f"Glu {row['E']*100:.1f}, Phe {row['F']*100:.1f}, Gly {row['G']*100:.1f}, His {row['H']*100:.1f}, "
            f"Ile {row['I']*100:.1f}, Lys {row['K']*100:.1f}, Leu {row['L']*100:.1f}, Met {row['M']*100:.1f}, "
            f"Asn {row['N']*100:.1f}, Pro {row['P']*100:.1f}, Gln {row['Q']*100:.1f}, Arg {row['R']*100:.1f}, "
            f"Ser {row['S']*100:.1f}, Thr {row['T']*100:.1f}, Val {row['V']*100:.1f}, Trp {row['W']*100:.1f}, "
            f"Tyr {row['Y']*100:.1f}. "

            # Amino acid property categories
            f"Amino acid properties: Tiny {row['Tiny']} ({row['Tiny_perc']:.2f}%), Small {row['Small']} ({row['Small_perc']:.2f}%), "
            f"Aliphatic {row['Aliphatic']} ({row['Aliphatic_perc']:.2f}%), Aromatic {row['Aromatic']} ({row['Aromatic_perc']:.2f}%), "
            f"Non-polar {row['NonPolar']} ({row['NonPolar_perc']:.2f}%), Polar {row['Polar']} ({row['Polar_perc']:.2f}%), "
            f"Charged {row['Charged']} ({row['Charged_perc']:.2f}%), Basic {row['Basic']} ({row['Basic_perc']:.2f}%), "
            f"Acidic {row['Acidic']} ({row['Acidic_perc']:.2f}%). "

            # # PPI network features
            # f"PPI metrics: degree centrality {row['degree_centrality']:.6f}, betweenness {row['betweenness_centrality']:.6f}, "
            # f"load centrality {row['load_centrality']:.6f}, eigenvector {row['eigenvector_centrality']:.6f}, "
            # f"closeness {row['closeness_centrality']:.6f}, PageRank {row['pagerank']:.6f}."

            f"PPI metrics: degree centrality: {row['degree_centrality_bucket']}, "
            f"betweenness: {row['betweenness_centrality_bucket']}, "
            f"load centrality: {row['load_centrality_bucket']}, "
            f"eigenvector: {row['eigenvector_centrality_bucket']}, "
            f"closeness: {row['closeness_centrality']:.2f}, "
            f"PageRank: {row['pagerank_bucket']}."

        )
        #label = 1 if row['Essential'] == 'essential' else 0
        label = row['Essential']
        serialized_texts.append(text)
        labels.append(label)

    return pd.DataFrame({'text': serialized_texts, 'essentiality': labels})


serialized_df = serialize_for_roberta(data_balanced) #try with sampled_data
print('Done.')

# serialized_df.to_csv("serialized_data.csv", index=False)

Done.


CV for roberta-base

In [ ]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


In [ ]:
from datasets import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
import evaluate
from sklearn.metrics import roc_auc_score
from scipy.special import softmax
#import shutil, glob, os

# 1️⃣ Dataset
dataset = Dataset.from_pandas(serialized_df)

# 2️⃣ Tokenization
tokenizer = RobertaTokenizer.from_pretrained("roberta-base") #distilroberta-base

def tokenize_function_roberta(example):
    model_inputs = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    model_inputs["labels"] = int(example["essentiality"])
    return model_inputs

tokenized_datasets = dataset.map(tokenize_function_roberta, batched=False, load_from_cache_file=True)

# 3️⃣ Prepare label array for StratifiedKFold
labels = [int(x) for x in tokenized_datasets["labels"]]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 4️⃣ Metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=preds, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"]
    }


fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    print(f"\n===== Fold {fold} =====")

    train_subset = Subset(tokenized_datasets, train_idx)
    val_subset = Subset(tokenized_datasets, val_idx)

    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
    #model = AutoModelForSequenceClassification.from_pretrained("distilroberta-base", num_labels=2)

    #training arguments for the general eukaryotic/bacterial model. find more info in word document of parameters
    training_args = TrainingArguments(
        #output_dir="bact_roberta_base_essentiality",
        #push_to_hub=True, # upload automatically
        #hub_model_id="sms112/bact_roberta_base_essentiality", # to push to HF

        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        per_device_train_batch_size=60,
        per_device_eval_batch_size=60,
        gradient_accumulation_steps=4,
        num_train_epochs=10,
        warmup_steps=0.1,
        weight_decay=0.01,
        seed=SEED,                      # seed
        data_seed=SEED,                 # ensures dataloader seed
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        logging_steps=500,
        fp16=True,
        report_to=[]
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=val_subset,
        compute_metrics=compute_metrics
    )

    trainer.train()


    # Get metrics from Trainer evaluation
    eval_metrics = trainer.evaluate()

    # Compute ROC-AUC manually
    preds = trainer.predict(val_subset)
    logits = preds.predictions
    true_labels = preds.label_ids
    probs = softmax(logits, axis=1)
    y_scores = probs[:, 1]
    roc_auc = roc_auc_score(true_labels, y_scores)

    eval_metrics["eval_roc_auc"] = roc_auc
    fold_results.append(eval_metrics)

    print(f"Fold {fold} results:")
    for k, v in eval_metrics.items():
        if "eval_" in k:
            print(f"  {k}: {v:.4f}")



# 6️⃣ Compute mean metrics across folds
metrics_to_average = ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1", "eval_roc_auc"]
mean_results = {metric: np.mean([f[metric] for f in fold_results]) for metric in metrics_to_average}

print("\n===== Mean CV Results across all 5 folds =====")
for metric, value in mean_results.items():
    print(f"{metric}: {value:.4f}")


Map:   0%|          | 0/14078 [00:00<?, ? examples/s]


===== Fold 1 =====


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.692128,0.500000,0.500000,1.000000,0.666667
2,No log,0.513939,0.765625,0.758287,0.779830,0.768908
3,No log,0.479787,0.780540,0.769074,0.801847,0.785118
4,No log,0.475540,0.778409,0.760638,0.812500,0.785714
5,No log,0.472306,0.777344,0.770617,0.789773,0.780077
6,No log,0.464457,0.780540,0.773925,0.792614,0.783158
7,No log,0.468162,0.778764,0.758734,0.817472,0.787009
8,No log,0.470505,0.776634,0.749200,0.831676,0.788287
9,No log,0.465063,0.782670,0.772230,0.801847,0.786760
10,No log,0.462322,0.781250,0.764352,0.813210,0.788025


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Fold 5 results:
  eval_loss: 0.4510
  eval_accuracy: 0.8028
  eval_precision: 0.7826
  eval_recall: 0.8388
  eval_f1: 0.8097
  eval_runtime: 4.6452
  eval_samples_per_second: 606.0060
  eval_steps_per_second: 10.1180
  eval_roc_auc: 0.8719

===== Mean CV Results across all 5 folds =====
eval_accuracy: 0.7911
eval_precision: 0.7742
eval_recall: 0.8217
eval_f1: 0.7972
eval_roc_auc: 0.8682


In [ ]:
# push to HF
trainer.push_to_hub("sms112/bact_roberta_base_essentiality")
tokenizer.push_to_hub("sms112/bact_roberta_base_essentiality")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tiality/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...tiality/model.safetensors:   8%|8         | 41.8MB /  499MB            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/sms112/bact_roberta_base_essentiality/commit/838b987beb8a4946cf2f373627ce229f8c5a1fc1', commit_message='Upload tokenizer', commit_description='', oid='838b987beb8a4946cf2f373627ce229f8c5a1fc1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sms112/bact_roberta_base_essentiality', endpoint='https://huggingface.co', repo_type='model', repo_id='sms112/bact_roberta_base_essentiality'), pr_revision=None, pr_num=None)

CV for roberta-large

In [ ]:
from datasets import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Subset
import evaluate
from sklearn.metrics import roc_auc_score
from scipy.special import softmax
#import shutil, glob, os

# 1️⃣ Dataset
dataset = Dataset.from_pandas(serialized_df)

# 2️⃣ Tokenization
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize_function_roberta(example):
    model_inputs = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    model_inputs["labels"] = int(example["essentiality"])
    return model_inputs

tokenized_datasets = dataset.map(tokenize_function_roberta, batched=False, load_from_cache_file=True)

# 3️⃣ Prepare label array for StratifiedKFold
labels = [int(x) for x in tokenized_datasets["labels"]]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 4️⃣ Metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=preds, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"]
    }



fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    print(f"\n===== Fold {fold} =====")

    train_subset = Subset(tokenized_datasets, train_idx)
    val_subset = Subset(tokenized_datasets, val_idx)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

    training_args = TrainingArguments(
        #output_dir="bact_roberta_large_essentiality",
        #push_to_hub=True, # upload automatically
        #hub_model_id="sms112/bact_roberta_large_essentiality", # to push to HF

        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        per_device_train_batch_size=50,
        per_device_eval_batch_size=50,
        gradient_accumulation_steps=4,
        num_train_epochs=10,
        weight_decay=0.01,
        seed=SEED,                      # seed
        data_seed=SEED,                 # ensures dataloader seed
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        logging_steps=500,
        fp16=True,
        report_to=[]
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        eval_dataset=val_subset,
        #tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()


    # Get metrics from Trainer evaluation
    eval_metrics = trainer.evaluate()

    # Compute ROC-AUC manually
    preds = trainer.predict(val_subset)
    logits = preds.predictions
    true_labels = preds.label_ids
    probs = softmax(logits, axis=1)
    y_scores = probs[:, 1]
    roc_auc = roc_auc_score(true_labels, y_scores)

    eval_metrics["eval_roc_auc"] = roc_auc
    fold_results.append(eval_metrics)

    print(f"Fold {fold} results:")
    for k, v in eval_metrics.items():
        if "eval_" in k:
            print(f"  {k}: {v:.4f}")



# 6️⃣ Compute mean metrics across folds
metrics_to_average = ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1", "eval_roc_auc"]
mean_results = {metric: np.mean([f[metric] for f in fold_results]) for metric in metrics_to_average}

print("\n===== Mean CV Results across all 5 folds =====")
for metric, value in mean_results.items():
    print(f"{metric}: {value:.4f}")


Map:   0%|          | 0/14078 [00:00<?, ? examples/s]


===== Fold 1 =====


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.492485,0.763494,0.773196,0.745739,0.759219
2,No log,0.470216,0.779830,0.775524,0.787642,0.781536
3,No log,0.466584,0.781250,0.784892,0.774858,0.779843
4,No log,0.472066,0.777699,0.740764,0.854403,0.793536
5,No log,0.473741,0.781250,0.763298,0.815341,0.788462
6,No log,0.456835,0.784091,0.754130,0.843040,0.796110
7,No log,0.456092,0.788352,0.749386,0.866477,0.803689
8,No log,0.445420,0.786932,0.757325,0.844460,0.798522
9,1.904262,0.440224,0.792969,0.763578,0.848722,0.803902
10,1.904262,0.440579,0.789418,0.755166,0.856534,0.802662


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Fold 5 results:
  eval_loss: 0.4380
  eval_accuracy: 0.8032
  eval_precision: 0.7795
  eval_recall: 0.8459
  eval_f1: 0.8113
  eval_runtime: 9.5131
  eval_samples_per_second: 295.9080
  eval_steps_per_second: 5.9920
  eval_roc_auc: 0.8774

===== Mean CV Results across all 5 folds =====
eval_accuracy: 0.7906
eval_precision: 0.7620
eval_recall: 0.8457
eval_f1: 0.8016
eval_roc_auc: 0.8703


In [ ]:
# push to HF
trainer.push_to_hub("sms112/bact_roberta_large_essentiality")
tokenizer.push_to_hub("sms112/bact_roberta_large_essentiality")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tiality/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...tiality/model.safetensors:   3%|2         | 41.8MB / 1.42GB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/sms112/bact_roberta_large_essentiality/commit/9c9541d0d7e711e9fd738cd4cb630cca83a74da2', commit_message='Upload tokenizer', commit_description='', oid='9c9541d0d7e711e9fd738cd4cb630cca83a74da2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sms112/bact_roberta_large_essentiality', endpoint='https://huggingface.co', repo_type='model', repo_id='sms112/bact_roberta_large_essentiality'), pr_revision=None, pr_num=None)

In [ ]:
#users load models from:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("sms112/euk_roberta_large_essentiality")
model = AutoModelForSequenceClassification.from_pretrained("sms112/euk_roberta_large_essentiality")